# Lesson 3: Reflection and Blogpost Writing

## Setup

In [1]:
llm_config = {"model": "gpt-3.5-turbo"}

## The task!

In [2]:
task = '''
        Write a concise but engaging blogpost about
       DeepLearning.AI. Make sure the blogpost is
       within 100 words.
       '''


## Create a writer agent

In [3]:
import autogen

writer = autogen.AssistantAgent(
    name="Writer",
    system_message="You are a writer. You write engaging and concise " 
        "blogpost (with title) on given topics. You must polish your "
        "writing based on the feedback you receive and give a refined "
        "version. Only return your final work without additional comments.",
    llm_config=llm_config,
)

In [4]:
reply = writer.generate_reply(messages=[{"content": task, "role": "user"}])

In [5]:
print(reply)

Title: "Demystifying DeepLearning.AI: Your Gateway to Artificial Intelligence"

Unlock the power of Artificial Intelligence with DeepLearning.AI! Founded by Andrew Ng, this platform offers top-tier courses in deep learning, neural networks, and machine learning. Whether you're a beginner or a seasoned pro, DeepLearning.AI has something for everyone. Dive into interactive assignments, gain real-world skills, and join a global community of learners. With engaging content and expert-led instruction, you'll be on your way to mastering AI in no time. Don't miss this opportunity to upskill and stay ahead in the rapidly evolving tech industry with DeepLearning.AI! 🚀🧠


## Adding reflection 

Create a critic agent to reflect on the work of the writer agent.

In [6]:
critic = autogen.AssistantAgent(
    name="Critic",
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    llm_config=llm_config,
    system_message="You are a critic. You review the work of "
                "the writer and provide constructive "
                "feedback to help improve the quality of the content.",
)

In [7]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
       DeepLearning.AI. Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

Title: "Demystifying DeepLearning.AI: Your Gateway to Artificial Intelligence"

Unlock the power of Artificial Intelligence with DeepLearning.AI! Founded by Andrew Ng, this platform offers top-tier courses in deep learning, neural networks, and machine learning. Whether you're a beginner or a seasoned pro, DeepLearning.AI has something for everyone. Dive into interactive assignments, gain real-world skills, and join a global community of learners. With engaging content and expert-led instruction, you'll be on your way to mastering AI in no time. Don't miss this opportunity to upskill and stay ahead in the rapidly evolving tech industry with DeepLearning.AI! 🚀🧠

----------------------------------------------------------------------

## Nested chat

In [8]:
SEO_reviewer = autogen.AssistantAgent(
    name="SEO Reviewer",
    llm_config=llm_config,
    system_message="You are an SEO reviewer, known for "
        "your ability to optimize content for search engines, "
        "ensuring that it ranks well and attracts organic traffic. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)


In [9]:
legal_reviewer = autogen.AssistantAgent(
    name="Legal Reviewer",
    llm_config=llm_config,
    system_message="You are a legal reviewer, known for "
        "your ability to ensure that content is legally compliant "
        "and free from any potential legal issues. "
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

In [10]:
ethics_reviewer = autogen.AssistantAgent(
    name="Ethics Reviewer",
    llm_config=llm_config,
    system_message="You are an ethics reviewer, known for "
        "your ability to ensure that content is ethically sound "
        "and free from any potential ethical issues. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role. ",
)

In [11]:
meta_reviewer = autogen.AssistantAgent(
    name="Meta Reviewer",
    llm_config=llm_config,
    system_message="You are a meta reviewer, you aggragate and review "
    "the work of other reviewers and give a final suggestion on the content.",
)

## Orchestrate the nested chats to solve the task

In [12]:
def reflection_message(recipient, messages, sender, config):
    return f'''Review the following content. 
            \n\n {recipient.chat_messages_for_summary(sender)[-1]['content']}'''

review_chats = [
    {
     "recipient": SEO_reviewer, 
     "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}. Here Reviewer should be your role",},
     "max_turns": 1},
    {
    "recipient": legal_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}.",},
     "max_turns": 1},
    {"recipient": ethics_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'reviewer': '', 'review': ''}",},
     "max_turns": 1},
     {"recipient": meta_reviewer, 
      "message": "Aggregrate feedback from all reviewers and give final suggestions on the writing.", 
     "max_turns": 1},
]


In [13]:
critic.register_nested_chats(
    review_chats,
    trigger=writer,
)

**Note**: You might get a slightly different response than what's shown in the video. Feel free to try different task.

In [14]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
       DeepLearning.AI. Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

Title: "Demystifying DeepLearning.AI: Your Gateway to Artificial Intelligence"

Unlock the power of Artificial Intelligence with DeepLearning.AI! Founded by Andrew Ng, this platform offers top-tier courses in deep learning, neural networks, and machine learning. Whether you're a beginner or a seasoned pro, DeepLearning.AI has something for everyone. Dive into interactive assignments, gain real-world skills, and join a global community of learners. With engaging content and expert-led instruction, you'll be on your way to mastering AI in no time. Don't miss this opportunity to upskill and stay ahead in the rapidly evolving tech industry with DeepLearning.AI! 🚀🧠

----------------------------------------------------------------------


--------------------------------------------------------------------------------
Critic (to Writer):

Aggregated feedback from all reviewers suggests that the content should focus on including relevant keywords, utilizing meta tags effectively, creating a call-to-action, and considering structured data markup to optimize the content for search engines and enhance user engagement.

Final suggestion on the writing:
- Ensure that relevant keywords are incorporated naturally throughout the content to improve search engine visibility.
- Pay close attention to meta tags to enhance the website's performance in search results.
- Craft a compelling call-to-action to encourage user interaction and engagement.
- Implement structured data markup to provide search engines with more context about your content.

Overall, the content should prioritize these SEO strategies to improve visibility, user engagement, and overall website performance.

--------------------------------------------------------

## Get the summary

In [15]:
print(res.summary)

Title: "Master AI Skills with DeepLearning.AI: Your Path to Success in Artificial Intelligence"

Embark on your AI learning journey with DeepLearning.AI, the leading platform by Andrew Ng. Discover comprehensive courses in deep learning, neural networks, and machine learning for all skill levels. Through interactive assignments and expert-led insights, equip yourself with practical AI knowledge. Join a vibrant global community of learners to enhance your skills and knowledge. Stay ahead in the tech industry by upskilling with DeepLearning.AI today! Don't miss the chance to delve into the world of AI. Start your learning adventure now.
